In [ ]:
import numpy as np
import torch
import time
from scipy import sparse as sp
from scipy.linalg import eigh
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

pauli_matrices = {
    1: np.array([[0, 1], [1, 0]], dtype=complex),
    2: np.array([[0, -1j], [1j, 0]], dtype=complex),
    3: np.array([[1, 0], [0, -1]], dtype=complex),
    0: np.eye(2, dtype=complex)
}

pauli_matrices_sparse = {
    key: sp.csr_matrix(value) for key, value in pauli_matrices.items()
}

def pauli_site(x, i, L):
    if L == 1:
        return pauli_matrices_sparse[x]
    if i == 1:
        return sp.kron(pauli_matrices_sparse[x], sp.eye(2**(L-1), format='csr'))
    else:
        return sp.kron(sp.eye(2), pauli_site(x, i-1, L-1), format='csr')

def pauli_int(spin, site, L):
    if site == L:  
        op = sp.eye(2**L, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == L or i == 1:
                op = op @ pauli_site(spin, i, L)
        return op
    else:
        op = sp.eye(2**L, dtype=complex, format='csr')
        for i in range(1, L+1):
            if i == site or i == site+1:
                op = op @ pauli_site(spin, i, L)
        return op

def xxx_hamiltonian_sparse(L, delta=1):
    dim = 2**L  
    H = sp.csr_matrix((dim, dim), dtype=complex)  
    for site in range(1, L+1):
        H += pauli_int(1, site, L)
        H += pauli_int(2, site, L)
        H += delta * pauli_int(3, site, L)
    return -H  

def build_full_hamiltonian_sparse(L, delta=1, Q=np.pi, h=0.5):
    H0 = xxx_hamiltonian_sparse(L, delta)   
    
    dim = 2**L
    Hz = sp.csr_matrix((dim, dim), dtype=complex)
    for n in range(1, L+1):
        Hz += np.cos(Q * n) * pauli_site(3, n, L)
    
    H = H0 + h * Hz  
    return -H       

def local_ops(dtype=torch.complex64, device="cpu"):
    Sz = torch.tensor([[0.5, 0.0],
                       [0.0, -0.5]], dtype=dtype, device=device)
    Sp = torch.tensor([[0.0, 1.0],
                       [0.0, 0.0]], dtype=dtype, device=device)
    Sm = torch.tensor([[0.0, 0.0],
                       [1.0, 0.0]], dtype=dtype, device=device)
    Sx = torch.tensor([[0.0, 1.0],
                       [1.0, 0.0]], dtype=dtype, device=device)
    Sy = torch.tensor([[0.0, -1j],
                       [1j, 0.0]], dtype=dtype, device=device)
    I2 = torch.eye(2, dtype=dtype, device=device)
    return I2, Sx, Sy, Sz, Sp, Sm

def precompute_perms(L: int):
    perms = []
    inv_perms = []
    axes = list(range(L))
    for s in range(L):
        p = [s] + [a for a in axes if a != s]
        inv = np.argsort(p)
        perms.append(p)
        inv_perms.append(inv.tolist())
    return perms, inv_perms

def apply_one_site(op2, site, vec, L, perms, inv_perms):
    psi = vec.reshape([2]*L).permute(perms[site]).reshape(2, -1)
    psi2 = op2 @ psi
    out = psi2.reshape([2]*L).permute(inv_perms[site]).reshape(-1)
    return out

def B_action(u, psi, L, I2, Sx, Sy, Sz, perms, inv_perms, alpha, beta, gamma):
    dtype, device = psi.dtype, psi.device
    ii = torch.tensor(1j, dtype=dtype, device=device)

    V11 = psi.clone()
    V12 = torch.zeros_like(psi)
    V21 = torch.zeros_like(psi)
    V22 = psi.clone()

    for site in range(L):
        ZV11 = 2 * apply_one_site(Sz, site, V11, L, perms, inv_perms)
        ZV12 = 2 * apply_one_site(Sz, site, V12, L, perms, inv_perms)
        ZV21 = 2 * apply_one_site(Sz, site, V21, L, perms, inv_perms)
        ZV22 = 2 * apply_one_site(Sz, site, V22, L, perms, inv_perms)

        XV11 = apply_one_site(Sx, site, V11, L, perms, inv_perms)
        XV12 = apply_one_site(Sx, site, V12, L, perms, inv_perms)
        XV21 = apply_one_site(Sx, site, V21, L, perms, inv_perms)
        XV22 = apply_one_site(Sx, site, V22, L, perms, inv_perms)

        YV11 = apply_one_site(Sy, site, V11, L, perms, inv_perms)
        YV12 = apply_one_site(Sy, site, V12, L, perms, inv_perms)
        YV21 = apply_one_site(Sy, site, V21, L, perms, inv_perms)
        YV22 = apply_one_site(Sy, site, V22, L, perms, inv_perms)

        L11V11 = u * V11 + (ii / 2) * gamma * ZV11
        L11V12 = u * V12 + (ii / 2) * gamma * ZV12
        L22V21 = u * V21 - (ii / 2) * gamma * ZV21
        L22V22 = u * V22 - (ii / 2) * gamma * ZV22

        L12V21 = (ii / 2) * (alpha * XV21 - ii * beta * YV21)
        L12V22 = (ii / 2) * (alpha * XV22 - ii * beta * YV22)
        L21V11 = (ii / 2) * (alpha * XV11 + ii * beta * YV11)
        L21V12 = (ii / 2) * (alpha * XV12 + ii * beta * YV12)

        N11 = L11V11 + L12V21
        N12 = L11V12 + L12V22
        N21 = L21V11 + L22V21
        N22 = L21V12 + L22V22

        V11, V12, V21, V22 = N11, N12, N21, N22

    return V12

def vacuum_state(L, dtype=torch.complex128, device="cpu"):
    dim = 2**L
    v = torch.zeros(dim, dtype=dtype, device=device)
    v[0] = 1.0 + 0.0j
    return v

def bethe_state_multilayer(u_list, L, I2, Sx, Sy, Sz, perms, inv_perms,
                           alpha, beta, gamma, normalize=True):
    psi = vacuum_state(L, dtype=I2.dtype, device=I2.device)
    for u in u_list:
        psi = B_action(u, psi, L, I2, Sx, Sy, Sz, perms, inv_perms,
                       alpha, beta, gamma)
    if normalize:
        norm = torch.sqrt(torch.real(torch.vdot(psi, psi)))
        if norm > 1e-15:
            psi = psi / norm
    return psi

def exact_diagonalization(H, num_states=5):
    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    eigenvalues, eigenvectors = eigh(H_dense)
    results = []
    for i in range(min(num_states, len(eigenvalues))):
        results.append({
            'energy': eigenvalues[i],
            'state': eigenvectors[:, i]
        })
    return results

def pytorch_loss_func(params, H_tensor, L, I2, Sx, Sy, Sz, perms, inv_perms, M_number=3):
    params_per_layer = 2
    total_u_params = M_number * params_per_layer

    u_list = []
    for layer in range(M_number):
        re = params[layer * 2]
        im = params[layer * 2 + 1]
        u = torch.complex(re, im)
        u_list.append(u)

    alpha = torch.complex(params[total_u_params],     params[total_u_params + 1])
    beta  = torch.complex(params[total_u_params + 2], params[total_u_params + 3])
    gamma = torch.complex(params[total_u_params + 4], params[total_u_params + 5])

    psi = bethe_state_multilayer(u_list, L, I2, Sx, Sy, Sz, perms, inv_perms,
                                 alpha, beta, gamma, normalize=True)

    H_psi = H_tensor @ psi
    energy = torch.real(torch.vdot(psi, H_psi))
    norm2 = torch.real(torch.vdot(psi, psi))
    energy_expectation = energy / norm2
    return energy_expectation

def torch_wrapper_loss(params_numpy, H_tensor, L, I2, Sx, Sy, Sz,
                       perms, inv_perms, M_number):
    params = torch.tensor(params_numpy, dtype=torch.float64, requires_grad=True)
    loss = pytorch_loss_func(params, H_tensor, L, I2, Sx, Sy, Sz,
                             perms, inv_perms, M_number)
    loss.backward()
    loss_value = loss.item()
    grad_value = params.grad.numpy().astype(np.float64)
    return loss_value, grad_value

def optimize_ground_state(L=6, delta=1.0, Q=np.pi, h=0.5, M_number=3,
                         num_attempts=5, maxiter=6000, device="cpu"):
    print(f"{L} sites")
    print(f"Anisotropy parameter Δ={delta}")
    print(f"Longitudinal field parameters Q={Q:.4f}, h={h:.4f}")
    print(f"Number of Bethe layers (M_number): {M_number}")
    print(f"Number of random initialization attempts: {num_attempts}")
    print(f"Device: {device}")

    H = build_full_hamiltonian_sparse(L, delta=delta, Q=Q, h=h)

    exact_states = exact_diagonalization(H, num_states=5)
    ground_energy = exact_states[0]['energy']
    excited1_energy = exact_states[1]['energy']
    print(f"Exact ground state energy: {ground_energy:.8f}")
    print(f"Exact first excited state energy: {excited1_energy:.8f}")

    I2, Sx, Sy, Sz, Sp, Sm = local_ops(dtype=torch.complex128, device=device)
    perms, inv_perms = precompute_perms(L)

    if sp.issparse(H):
        H_dense = H.toarray()
    else:
        H_dense = H
    H_tensor = torch.tensor(H_dense, dtype=torch.complex128, device=device)
    ground_state_tensor = torch.tensor(exact_states[0]['state'],
                                       dtype=torch.complex128, device=device)

    best_energy = float('inf')
    best_params = None
    best_psi = None
    best_fidelity = 0.0

    params_per_layer = 2
    total_u_params = M_number * params_per_layer
    total_params = total_u_params + 6

    for attempt in range(num_attempts):
        print(f"  Attempt {attempt+1}/{num_attempts}:")
        x0 = np.zeros(total_params, dtype=np.float64)

        for layer in range(M_number):
            start_idx = layer * 2
            x0[start_idx:start_idx+2] = np.random.normal(-100, 100, 2)

        x0[total_u_params:total_u_params+6] = np.random.uniform(-10, 10, 6)

        bounds = [(-3, 3)] * total_u_params + [(-5, 5)] * 6

        def loss_func_wrapper(p):
            return torch_wrapper_loss(p, H_tensor, L, I2, Sx, Sy, Sz,
                                      perms, inv_perms, M_number)

        try:
            result = minimize(
                lambda p: loss_func_wrapper(p)[0],
                x0,
                method='L-BFGS-B',
                jac=lambda p: loss_func_wrapper(p)[1],
                options={'maxiter': maxiter, 'ftol': 1e-12, 'gtol': 1e-10},
                bounds=bounds
            )

            final_params = result.x
            final_loss = result.fun

            with torch.no_grad():
                u_list_tensor = []
                for layer in range(M_number):
                    re = final_params[layer * 2]
                    im = final_params[layer * 2 + 1]
                    u = torch.complex(torch.tensor(re, dtype=torch.float64),
                                      torch.tensor(im, dtype=torch.float64)).to(device)
                    u_list_tensor.append(u)

                alpha_tensor = torch.complex(
                    torch.tensor(final_params[total_u_params], dtype=torch.float64),
                    torch.tensor(final_params[total_u_params+1], dtype=torch.float64)
                ).to(device)
                beta_tensor = torch.complex(
                    torch.tensor(final_params[total_u_params+2], dtype=torch.float64),
                    torch.tensor(final_params[total_u_params+3], dtype=torch.float64)
                ).to(device)
                gamma_tensor = torch.complex(
                    torch.tensor(final_params[total_u_params+4], dtype=torch.float64),
                    torch.tensor(final_params[total_u_params+5], dtype=torch.float64)
                ).to(device)

                psi = bethe_state_multilayer(u_list_tensor, L, I2, Sx, Sy, Sz,
                                             perms, inv_perms,
                                             alpha_tensor, beta_tensor, gamma_tensor,
                                             normalize=True)

            psi_numpy = psi.cpu().detach().numpy()
            norm = np.vdot(psi_numpy, psi_numpy)

            if sp.issparse(H):
                H_psi = H.dot(psi_numpy)
            else:
                H_psi = H @ psi_numpy

            energy = np.real(np.vdot(psi_numpy, H_psi) / norm)

            fidelity = 0.0
            for idx in [1, 2]:   
                overlap = np.abs(np.vdot(psi_numpy, exact_states[idx]['state']))**2 / norm
                fidelity += overlap

            print(f"    Loss value: {final_loss:.8f}")
            print(f"    Energy expectation: {energy:.8f}")
            print(f"    Fidelity (1st+2nd excited states): {fidelity:.8f}")
            print(f"    Parameters of this attempt:")
            for layer in range(M_number):
                re_part = final_params[layer*2]
                im_part = final_params[layer*2+1]
                u = re_part + 1j*im_part
                print(f"      Layer {layer+1}: u = {u.real:.6f} + {u.imag:.6f}i")
            alpha_val = final_params[total_u_params] + 1j*final_params[total_u_params+1]
            beta_val  = final_params[total_u_params+2] + 1j*final_params[total_u_params+3]
            gamma_val = final_params[total_u_params+4] + 1j*final_params[total_u_params+5]
            print(f"      α = {alpha_val.real:.6f} + {alpha_val.imag:.6f}i")
            print(f"      β = {beta_val.real:.6f} + {beta_val.imag:.6f}i")
            print(f"      γ = {gamma_val.real:.6f} + {gamma_val.imag:.6f}i")
            
            if energy < best_energy:
                best_energy = energy
                best_params = final_params
                best_psi = psi_numpy
                best_fidelity = fidelity

        except Exception as e:
            print(f"    Optimization failed: {e}")
            continue

    return best_energy, best_params, best_psi, exact_states, best_fidelity

if __name__ == "__main__":
    L = 6
    delta = 1.0
    Q = np.pi          
    h = 0.1           
    M_number = 2
    num_attempts = 10
    device = "cuda" if torch.cuda.is_available() else "cpu"

    start_time = time.time()

    best_energy, best_params, best_psi, exact_states, best_fidelity = optimize_ground_state(
        L=L,
        delta=delta,
        Q=Q,
        h=h,
        M_number=M_number,
        num_attempts=num_attempts,
        maxiter=5000,
        device=device
    )

    end_time = time.time()
    elapsed_time = end_time - start_time

    print("Optimization results")

    if best_params is not None:
        print(f"\nBethe ansatz ground state energy: {best_energy:.8f}")
        print(f"Exact ground state energy:        {exact_states[0]['energy']:.8f}")
        print(f"Energy difference:                 {abs(best_energy - exact_states[0]['energy']):.8f}")

        print(f"\nFidelity (1st+2nd excited states): {best_fidelity:.8f}")

        for i in [1, 2]:
            if i < len(exact_states):
                overlap = np.abs(np.vdot(best_psi, exact_states[i]['state']))
                print(f"Overlap with excited state {i}:      {overlap:.8f}")

        total_u_params = M_number * 2
        alpha = best_params[total_u_params] + 1j * best_params[total_u_params+1]
        beta  = best_params[total_u_params+2] + 1j * best_params[total_u_params+3]
        gamma = best_params[total_u_params+4] + 1j * best_params[total_u_params+5]
        print(f"\nOptimized shared parameters:")
        print(f"  α = {alpha.real:.6f} + {alpha.imag:.6f}i")
        print(f"  β = {beta.real:.6f} + {beta.imag:.6f}i")
        print(f"  γ = {gamma.real:.6f} + {gamma.imag:.6f}i")

        print(f"\nOptimized layer parameters (u):")
        params_per_layer = 2
        for layer in range(M_number):
            start_idx = layer * params_per_layer
            re_part = best_params[start_idx]
            im_part = best_params[start_idx + 1]
            u = re_part + 1j * im_part
            print(f"  Layer {layer+1}: u = {u.real:.6f} + {u.imag:.6f}i")

        print(f"\nOptimization time:              {elapsed_time:.2f} seconds")
    else:
        print("Optimization failed, no valid solution found.")